# Output 7 — Risk rating summary

Edit CI Summary, Input 6/7, and Macro/Ext in Excel, save, then reload.

Mechanical ratings come from Chart Data (baseline + standard B-tests).
Finals default to mechanical until judgement is applied.

See `docs/07-risk-summary.qmd`. For Output 5-1 / 5-2 see `demo/output_5.ipynb`.


In [ ]:
from __future__ import annotations

from pathlib import Path

from lic_dsf.load import load_core, load_rating, load_stress
from lic_dsf.output import risk_summary_panel
from lic_dsf.rating import (
    ChartDataRegistry,
    RiskRatingSummary,
    compute_mechanical_ratings,
    moderate_panel,
)
from lic_dsf.stress import run_b1_gdp_public, run_standard_external_stress

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "demo":
    REPO_ROOT = REPO_ROOT.parent
WORKBOOK = REPO_ROOT / "data" / "lic-dsf-template-2025-08-12.xlsx"
WORKBOOK


In [ ]:
# Quick path: compute Output 7.
from lic_dsf.run import SHEET_7, compute_outputs

output_7 = compute_outputs(WORKBOOK, include=[SHEET_7])
output_7.sheets[SHEET_7]

In [ ]:
macro, external, ext_base, pub_base = load_core(WORKBOOK)
rating = load_rating(WORKBOOK)
stress = load_stress(WORKBOOK)
ci = rating.ci

external_stress = run_standard_external_stress(
    macro, external, stress.input6, stress.residual
)
public_b1 = run_b1_gdp_public(macro, external, stress.input6, stress.residual)

first = int(macro.inputs.first_projection_year)
proj_years = list(range(first, first + 11))
ci.country, ci.dcc.value, round(ci.ci_score, 4)


## Chart Data → mechanical ratings → Output 7


In [ ]:
registry = ChartDataRegistry()
_EXTERNAL = (
    ("pv_debt_to_gdp", "pv_ppg_external_to_gdp"),
    ("pv_debt_to_exports", "pv_ppg_external_to_exports"),
    ("debt_service_to_exports", "ppg_debt_service_to_exports"),
    ("debt_service_to_revenue", "ppg_debt_service_to_revenue"),
)
for indicator, method in _EXTERNAL:
    registry.register_series(
        indicator,
        "baseline",
        getattr(ext_base, method)().reindex(proj_years),
        is_baseline=True,
    )
    for sid, book in external_stress.items():
        registry.register_series(
            indicator,
            sid,
            getattr(book, method)().reindex(proj_years),
            is_shock=True,
        )

registry.register_series(
    "public_pv_debt_to_gdp",
    "baseline",
    pub_base.pv_public_debt_to_gdp().reindex(proj_years),
    is_baseline=True,
)
registry.register_series(
    "public_pv_debt_to_gdp",
    "B1_GDP",
    public_b1.pv_public_debt_to_gdp().reindex(proj_years),
    is_shock=True,
)

mechanical = compute_mechanical_ratings(registry, ci.thresholds, years=proj_years)
out_5_1 = moderate_panel(
    mechanical_external=mechanical.external,
    baseline_pv_gdp=ext_base.pv_ppg_external_to_gdp(),
    threshold_pv_gdp=ci.thresholds.pv_debt_to_gdp,
    rating_years=proj_years,
)
summary = RiskRatingSummary(
    mechanical=mechanical,
    thresholds=ci.thresholds,
    dcc=ci.dcc,
    ci_score=ci.ci_score,
    moderate_granularity=str(out_5_1.loc["Space to absorb shock", "Output 5-1"]),
)
out_7 = risk_summary_panel(summary)
out_7
